# Cross-Dataset Testing

In [1]:
# Basic Imports
import sys
import pandas as pd
import numpy as np
import time
import joblib
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Importing shared functions
from src.model_saving import run_and_save_model
from src.model_testing import test_model

# Model-specific imports
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Loading and examining datasets
* Welfake (Original Dataset)
* IFND
* WWFND

In [2]:
df_raw = pd.read_csv("../data/Dataset.csv")

### Examining IFND dataset

In [3]:
IFND_raw = pd.read_csv("../data/IFND.csv", encoding="latin1")

In [4]:
# Examining Dataset 
print("Dataset shape:",IFND_raw.shape)
print("\nSample: ")
IFND_raw.head(1)

Dataset shape: (56714, 7)

Sample: 


,id,Statement,Image,Web,Category,Date,Label
0,2,"WHO praises India's Aarogya Setu app, says it ...",https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,COVID-19,Oct-20,TRUE


In [5]:
IFND_raw["title"] = ""
IFND_raw["text"] = IFND_raw["Statement"]

In [6]:
IFND_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56714 entries, 0 to 56713
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         56714 non-null  int64 
 1   Statement  56714 non-null  object
 2   Image      56714 non-null  object
 3   Web        56714 non-null  object
 4   Category   56714 non-null  object
 5   Date       45393 non-null  object
 6   Label      56714 non-null  object
 7   title      56714 non-null  object
 8   text       56714 non-null  object
dtypes: int64(1), object(8)
memory usage: 3.9+ MB


In [7]:
print(IFND_raw["Label"].unique())

['TRUE' 'Fake']


In [8]:
IFND_raw["label"] = (
    IFND_raw["Label"]
    .str.strip()
    .str.upper()
    .map({"TRUE": 1, "FAKE": 0})
)
print(IFND_raw["label"].isna().sum())
IFND_raw["label"] = IFND_raw["label"].astype(int)

0


In [9]:
print(IFND_raw["label"].value_counts())

label
1    37800
0    18914
Name: count, dtype: int64


In [10]:
IFND_raw.head(1)

,id,Statement,Image,Web,Category,Date,Label,title,text,label
0,2,"WHO praises India's Aarogya Setu app, says it ...",https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,COVID-19,Oct-20,TRUE,,"WHO praises India's Aarogya Setu app, says it ...",1


### Examining WWFND dataset

In [11]:
WWFND_true_raw = pd.read_csv("../data/WWFND_TrueNews_RawData.csv")
WWFND_fake_raw = pd.read_csv("../data/WWFND_FakeNews_RawData.csv")

WWFND_fake_raw["label"] = 0
WWFND_true_raw["label"] = 1
WWFND_raw = pd.concat([WWFND_fake_raw, WWFND_true_raw])

In [12]:
# Examining Dataset 
print("Dataset shape:",WWFND_raw.shape)
print("\nSample: ")
WWFND_raw.head(1)

Dataset shape: (30617, 10)

Sample: 


,Id,Category,Date,Label,Link,Originated_By,Poltifact_Label,Source,Text,label
0,0,Politics/War/Economy/Health etc.,"• March 21, 2022",Fake,https://www.politifact.com/factchecks/2022/mar...,Facebook posts,FALSE,Poltifact,Photos show “Ukrainian farmers are slowly star...,0


In [13]:
WWFND_raw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30617 entries, 0 to 15099
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Id               30617 non-null  int64 
 1   Category         30617 non-null  object
 2   Date             26730 non-null  object
 3   Label            30617 non-null  object
 4   Link             30411 non-null  object
 5   Originated_By    30617 non-null  object
 6   Poltifact_Label  14886 non-null  object
 7   Source           30617 non-null  object
 8   Text             30616 non-null  object
 9   label            30617 non-null  int64 
dtypes: int64(2), object(8)
memory usage: 2.6+ MB


In [14]:
WWFND_raw["title"] = ""
WWFND_raw["text"] = WWFND_raw["Text"]

## Training best models on complete WELFAKE dataset

### SVM: 
* Baseline
* No source words
* Text Only
* Welfake-tuned parameters ver

In [15]:
# SVM baseline

model_svm_baseline, vectorizer_title_svm_baseline, vectorizer_text_svm_baseline, feature_names_svm_baseline = run_and_save_model(
    df_raw,
    LinearSVC(random_state=42), 
    vectorizer="tfidf", 
    model_name="svm_baseline",
)


Features created in 51.9457 seconds.
Total features =  523007
Training completed in 1.2729 seconds.


In [16]:
# SVM No-sourcewords

model_svm_no_source , vectorizer_title_svm_no_source, vectorizer_text_svm_no_source, feature_names_svm_no_source = run_and_save_model(
    df_raw,
    LinearSVC(random_state=42, max_iter=5000), 
    vectorizer="tfidf", 
    model_name="svm_no_source_words",
    remove_sourcewords=True
)


Features created in 55.7825 seconds.
Total features =  522642
Training completed in 1.4417 seconds.


In [17]:
# SVM Text-only

model_svm_text_only, vectorizer_title_svm_text_only, vectorizer_text_svm_text_only, feature_names_svm_text_only = run_and_save_model(
    df_raw,
    LinearSVC(random_state=42, max_iter=5000), 
    vectorizer="tfidf", 
    model_name="svm_text_only",
    vec_title=False
)


Features created in 50.2505 seconds.
Total features =  500000
Training completed in 1.1351 seconds.


In [18]:
# SVM Title-only

model_svm_title_only, vectorizer_title_svm_title_only, vectorizer_text_svm_title_only, feature_names_svm_title_only = run_and_save_model(
    df_raw,
    LinearSVC(random_state=42, max_iter=5000), 
    vectorizer="tfidf", 
    model_name="svm_title_only",
    vec_text=False
)


Features created in 8.8964 seconds.
Total features =  23007
Training completed in 0.1576 seconds.


In [19]:
# SVM hyper-tuned

model_svm_hyper_tuned, vectorizer_title_svm_hyper_tuned, vectorizer_text_svm_hyper_tuned, feature_names_svm_hyper_tuned = run_and_save_model(
    df_raw,
    LinearSVC(random_state=42, max_iter=5000, loss='hinge', C=10), 
    vectorizer="tfidf", 
    model_name="svm_hyper_tuned",
    mdf=2
)


Features created in 50.8266 seconds.
Total features =  550000
Training completed in 16.3370 seconds.


### Logistic Regression: 
* Baseline
* No source words
* Text Only

In [20]:
# LR Baseline

model_lr_baseline, vectorizer_title_lr_baseline, vectorizer_text_lr_baseline, feature_names_lr_baseline = run_and_save_model(
    df_raw,
    LogisticRegression(random_state=42, max_iter=5000), 
    vectorizer="count", 
    model_name="lr_baseline",
)


Features created in 54.7059 seconds.
Total features =  523007
Training completed in 6.7484 seconds.


In [21]:
# LR No-sourcewords

model_lr_no_source, vectorizer_title_lr_no_source, vectorizer_text_lr_no_source, feature_names_lr_no_source = run_and_save_model(
    df_raw,
    LogisticRegression(random_state=42, max_iter=5000), 
    vectorizer="count", 
    model_name="lr_no_source_words",
    remove_sourcewords=True
)


Features created in 55.6933 seconds.
Total features =  522642
Training completed in 7.3813 seconds.


In [22]:
# LR Text-only

model_lr_text_only, vectorizer_title_lr_text_only, vectorizer_text_lr_text_only, feature_names_lr_text_only = run_and_save_model(
    df_raw,
    LogisticRegression(random_state=42, max_iter=5000), 
    vectorizer="count", 
    model_name="lr_text_only",
    vec_title=False
)


Features created in 52.1135 seconds.
Total features =  500000
Training completed in 7.7355 seconds.


In [23]:
# LR Title-only

model_lr_title_only, vectorizer_title_lr_title_only, vectorizer_text_lr_title_only, feature_names_lr_title_only = run_and_save_model(
    df_raw,
    LogisticRegression(random_state=42, max_iter=5000), 
    vectorizer="count", 
    model_name="lr_title_only",
    vec_text=False
)


Features created in 8.6422 seconds.
Total features =  23007
Training completed in 0.2455 seconds.


## Testing on different datasets

In [24]:
results = []

### IFND Dataset

In [25]:
# SVM Baseline
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_svm_baseline, 
    vectorizer_text_svm_baseline, 
    model_svm_baseline
)
results.append({
    "Dataset": "IFND",
    "Model": "SVM",
    "Version": "Baseline",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.1559 seconds.
Total features =  523007
Accuracy:  0.6643509539090877
Precision:  0.6663475177304965
Recall:  0.9942328042328042
F1:  0.4039523792487689

Confusion Matrix:
 [[   96 18818]
 [  218 37582]]

Top 5 Negative Features:
                   Feature  Importance
386358             reuters  -13.306503
2596             breitbart   -6.685861
395359                said   -6.621721
476805              trumps   -4.308603
499315  washington reuters   -3.911311

Top 5 Positive Features:
          Feature  Importance
490880        via    6.280664
226968      image    3.927209
21764       video    3.683415
227058  image via    3.536633
2542     breaking    3.035444


In [26]:
# SVM No-Source Words
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_svm_no_source, 
    vectorizer_text_svm_no_source, 
    model_svm_no_source
)
results.append({
    "Dataset": "IFND",
    "Model": "SVM",
    "Version": "No-Source Words",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.2241 seconds.
Total features =  522642
Accuracy:  0.6584793878054801
Precision:  0.6664559362751296
Recall:  0.9761111111111112
F1:  0.41815486838609994

Confusion Matrix:
 [[  448 18466]
 [  903 36897]]

Top 5 Negative Features:
                 Feature  Importance
395965              said   -7.538488
22643                000   -5.461247
187466            follow   -4.743824
354247  president donald   -4.686593
474698            trumps   -4.583563

Top 5 Positive Features:
          Feature  Importance
490416        via    7.515545
232757      image    4.460126
21429       video    4.415900
232847  image via    4.198588
318872    october    3.630076


In [27]:
# SVM Text-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_svm_text_only, 
    vectorizer_text_svm_text_only, 
    model_svm_text_only,
    text_only=True
)
results.append({
    "Dataset": "IFND",
    "Model": "SVM",
    "Version": "Text-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.1234 seconds.
Total features =  500000
Accuracy:  0.6635927636915048
Precision:  0.6665065727448992
Recall:  0.9912433862433863
F1:  0.407086159667873

Confusion Matrix:
 [[  166 18748]
 [  331 37469]]

Top 5 Negative Features:
        Feature  Importance
363351  reuters  -16.976054
1           000   -7.255445
372352     said   -6.798780
164494   follow   -5.275551
453798   trumps   -5.157739

Top 5 Positive Features:
          Feature  Importance
467873        via    8.935727
204051  image via    4.066633
203961      image    3.973319
296067    october    3.699707
195396    hillary    3.119853


In [28]:
# SVM Title-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_svm_title_only, 
    vectorizer_text_svm_title_only, 
    model_svm_title_only,
    title_only=True
)
results.append({
    "Dataset": "IFND",
    "Model": "SVM",
    "Version": "Title-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 0.3117 seconds.
Total features =  23007
Accuracy:  0.6665020982473463
Precision:  0.6665020982473463
Recall:  1.0
F1:  0.3999407495185898

Confusion Matrix:
 [[    0 18914]
 [    0 37800]]

Top 5 Negative Features:
          Feature  Importance
2596    breitbart  -10.322083
22909  york times   -6.945829
6976      factbox   -3.967580
5281   delingpole   -3.407945
16638    rohingya   -3.019260

Top 5 Positive Features:
        Feature  Importance
21764     video    7.176609
2542   breaking    4.720463
22783       wow    3.324654
20996    tweets    3.084638
22178     watch    2.878481


In [29]:
# SVM Hyper-Tuned
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_svm_hyper_tuned, 
    vectorizer_text_svm_hyper_tuned, 
    model_svm_hyper_tuned 
)
results.append({
    "Dataset": "IFND",
    "Model": "SVM",
    "Version": "Hyper-Tuned",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.3074 seconds.
Total features =  550000
Accuracy:  0.6653736290862926
Precision:  0.6665840620242858
Recall:  0.9962433862433863
F1:  0.4034444473465273

Confusion Matrix:
 [[   78 18836]
 [  142 37658]]

Top 5 Negative Features:
                   Feature  Importance
415624             reuters  -14.992427
5573             breitbart   -7.575153
425371                said   -7.037073
501706              trumps   -4.655935
525255  washington reuters   -4.373055

Top 5 Positive Features:
          Feature  Importance
516356        via    7.174090
255484      image    4.457442
46999       video    4.185202
255574  image via    4.027154
5453     breaking    3.335260


In [30]:
# LR Baseline
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_lr_baseline, 
    vectorizer_text_lr_baseline, 
    model_lr_baseline, 
)
results.append({
    "Dataset": "IFND",
    "Model": "LR",
    "Version": "Baseline",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.2767 seconds.
Total features =  523007
Accuracy:  0.6665549952392708
Precision:  0.6665373560684876
Recall:  1.0
F1:  0.40011203208646584

Confusion Matrix:
 [[    3 18911]
 [    0 37800]]

Top 5 Negative Features:
                   Feature  Importance
2596             breitbart   -5.165471
386358             reuters   -4.996144
23008                  000   -2.454076
499315  washington reuters   -2.069243
22909           york times   -1.813579

Top 5 Positive Features:
          Feature  Importance
21764       video    3.320407
490880        via    2.954029
2542     breaking    1.951130
227058  image via    1.613038
319074    october    1.213556


In [31]:
# LR No-Source Words
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_lr_no_source, 
    vectorizer_text_lr_no_source, 
    model_lr_no_source
)
results.append({
    "Dataset": "IFND",
    "Model": "LR",
    "Version": "No-Source Words",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.4998 seconds.
Total features =  522642
Accuracy:  0.6665197305779877
Precision:  0.6666137370101801
Recall:  0.9995502645502645
F1:  0.40085889268258984

Confusion Matrix:
 [[   18 18896]
 [   17 37783]]

Top 5 Negative Features:
                 Feature  Importance
22643                000   -2.694796
354247  president donald   -2.110442
187466            follow   -1.759944
16802               says   -1.747135
108851               com   -1.704954

Top 5 Positive Features:
              Feature  Importance
21429           video    3.807602
490416            via    3.205366
2500         breaking    2.402602
232847      image via    1.731367
315391  november 2016    1.372570


In [32]:
# LR Text-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_lr_text_only, 
    vectorizer_text_lr_text_only, 
    model_lr_text_only,
    text_only=True
)
results.append({
    "Dataset": "IFND",
    "Model": "LR",
    "Version": "Text-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 1.2385 seconds.
Total features =  500000
Accuracy:  0.6665726275699122
Precision:  0.6665491095044965
Recall:  1.0
F1:  0.4001691152776711

Confusion Matrix:
 [[    4 18910]
 [    0 37800]]

Top 5 Negative Features:
                   Feature  Importance
363351             reuters   -5.261452
1                      000   -3.034648
476308  washington reuters   -2.251313
164494              follow   -1.895461
87179                  com   -1.797532

Top 5 Positive Features:
              Feature  Importance
467873            via    3.280848
204051      image via    1.776547
393376          share    1.318721
296067        october    1.259612
292582  november 2016    1.232121


In [33]:
# LR Title-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    IFND_raw, 
    vectorizer_title_lr_title_only, 
    vectorizer_text_lr_title_only, 
    model_lr_title_only,
    title_only=True
)
results.append({
    "Dataset": "IFND",
    "Model": "LR",
    "Version": "Title-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  56714
No of empty text rows:  0
No of rows with empty title AND text:  0

Features created in 0.2766 seconds.
Total features =  23007
Accuracy:  0.6665020982473463
Precision:  0.6665020982473463
Recall:  1.0
F1:  0.3999407495185898

Confusion Matrix:
 [[    0 18914]
 [    0 37800]]

Top 5 Negative Features:
          Feature  Importance
2596    breitbart   -6.310209
22909  york times   -4.528080
6976      factbox   -3.645147
5281   delingpole   -3.226715
16638    rohingya   -2.981946

Top 5 Positive Features:
        Feature  Importance
2542   breaking    3.703229
21764     video    3.483429
22783       wow    2.852267
564      actual    2.726850
22914     youll    2.682271


### WWFND Dataset

In [34]:
# SVM Baseline
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_svm_baseline, 
    vectorizer_text_svm_baseline, 
    model_svm_baseline
)
results.append({
    "Dataset": "WWFND",
    "Model": "SVM",
    "Version": "Baseline",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.8982 seconds.
Total features =  523007
Accuracy:  0.5010615017800568
Precision:  0.497046583434018
Recall:  0.980794701986755
F1:  0.3623744034934845

Confusion Matrix:
 [[  531 14986]
 [  290 14810]]

Top 5 Negative Features:
                   Feature  Importance
386358             reuters  -13.306503
2596             breitbart   -6.685861
395359                said   -6.621721
476805              trumps   -4.308603
499315  washington reuters   -3.911311

Top 5 Positive Features:
          Feature  Importance
490880        via    6.280664
226968      image    3.927209
21764       video    3.683415
227058  image via    3.536633
2542     breaking    3.035444


In [35]:
# SVM No-Source Words
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_svm_no_source, 
    vectorizer_text_svm_no_source, 
    model_svm_no_source
)
results.append({
    "Dataset": "WWFND",
    "Model": "SVM",
    "Version": "No-Source Words",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.8536 seconds.
Total features =  522642
Accuracy:  0.5008328706274292
Precision:  0.4967666702003604
Recall:  0.9309933774834437
F1:  0.3954692298368869

Confusion Matrix:
 [[ 1276 14241]
 [ 1042 14058]]

Top 5 Negative Features:
                 Feature  Importance
395965              said   -7.538488
22643                000   -5.461247
187466            follow   -4.743824
354247  president donald   -4.686593
474698            trumps   -4.583563

Top 5 Positive Features:
          Feature  Importance
490416        via    7.515545
232757      image    4.460126
21429       video    4.415900
232847  image via    4.198588
318872    october    3.630076


In [36]:
# SVM Text-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_svm_text_only, 
    vectorizer_text_svm_text_only, 
    model_svm_text_only,
    text_only=True
)
results.append({
    "Dataset": "WWFND",
    "Model": "SVM",
    "Version": "Text-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.8162 seconds.
Total features =  500000
Accuracy:  0.5000489923898488
Precision:  0.4964990021310422
Recall:  0.9720529801324503
F1:  0.3667778355059386

Confusion Matrix:
 [[  632 14885]
 [  422 14678]]

Top 5 Negative Features:
        Feature  Importance
363351  reuters  -16.976054
1           000   -7.255445
372352     said   -6.798780
164494   follow   -5.275551
453798   trumps   -5.157739

Top 5 Positive Features:
          Feature  Importance
467873        via    8.935727
204051  image via    4.066633
203961      image    3.973319
296067    october    3.699707
195396    hillary    3.119853


In [37]:
# SVM Title-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_svm_title_only, 
    vectorizer_text_svm_title_only, 
    model_svm_title_only,
    title_only=True
)
results.append({
    "Dataset": "WWFND",
    "Model": "SVM",
    "Version": "Title-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.1828 seconds.
Total features =  23007
Accuracy:  0.49319005781102004
Precision:  0.49319005781102004
Recall:  1.0
F1:  0.33029288885972397

Confusion Matrix:
 [[    0 15517]
 [    0 15100]]

Top 5 Negative Features:
          Feature  Importance
2596    breitbart  -10.322083
22909  york times   -6.945829
6976      factbox   -3.967580
5281   delingpole   -3.407945
16638    rohingya   -3.019260

Top 5 Positive Features:
        Feature  Importance
21764     video    7.176609
2542   breaking    4.720463
22783       wow    3.324654
20996    tweets    3.084638
22178     watch    2.878481


In [38]:
# SVM Hyper-Tuned
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_svm_hyper_tuned, 
    vectorizer_text_svm_hyper_tuned, 
    model_svm_hyper_tuned 
)
results.append({
    "Dataset": "WWFND",
    "Model": "SVM",
    "Version": "Hyper-Tuned",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.9059 seconds.
Total features =  550000
Accuracy:  0.5001143155763138
Precision:  0.49658754203149447
Recall:  0.9878145695364239
F1:  0.35506145503838515

Confusion Matrix:
 [[  396 15121]
 [  184 14916]]

Top 5 Negative Features:
                   Feature  Importance
415624             reuters  -14.992427
5573             breitbart   -7.575153
425371                said   -7.037073
501706              trumps   -4.655935
525255  washington reuters   -4.373055

Top 5 Positive Features:
          Feature  Importance
516356        via    7.174090
255484      image    4.457442
46999       video    4.185202
255574  image via    4.027154
5453     breaking    3.335260


In [39]:
# LR Baseline
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_lr_baseline, 
    vectorizer_text_lr_baseline, 
    model_lr_baseline, 
)
results.append({
    "Dataset": "WWFND",
    "Model": "LR",
    "Version": "Baseline",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.9176 seconds.
Total features =  523007
Accuracy:  0.49354933533657774
Precision:  0.49336384439359265
Recall:  0.9994701986754967
F1:  0.3315353153628691

Confusion Matrix:
 [[   19 15498]
 [    8 15092]]

Top 5 Negative Features:
                   Feature  Importance
2596             breitbart   -5.165471
386358             reuters   -4.996144
23008                  000   -2.454076
499315  washington reuters   -2.069243
22909           york times   -1.813579

Top 5 Positive Features:
          Feature  Importance
21764       video    3.320407
490880        via    2.954029
2542     breaking    1.951130
227058  image via    1.613038
319074    october    1.213556


In [40]:
# LR No-Source Words
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_lr_no_source, 
    vectorizer_text_lr_no_source, 
    model_lr_no_source
)
results.append({
    "Dataset": "WWFND",
    "Model": "LR",
    "Version": "No-Source Words",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.8857 seconds.
Total features =  522642
Accuracy:  0.49361465852304276
Precision:  0.4933578850453768
Recall:  0.9936423841059603
F1:  0.3366042556656929

Confusion Matrix:
 [[  109 15408]
 [   96 15004]]

Top 5 Negative Features:
                 Feature  Importance
22643                000   -2.694796
354247  president donald   -2.110442
187466            follow   -1.759944
16802               says   -1.747135
108851               com   -1.704954

Top 5 Positive Features:
              Feature  Importance
21429           video    3.807602
490416            via    3.205366
2500         breaking    2.402602
232847      image via    1.731367
315391  november 2016    1.372570


In [41]:
# LR Text-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_lr_text_only, 
    vectorizer_text_lr_text_only, 
    model_lr_text_only,
    text_only=True
)
results.append({
    "Dataset": "WWFND",
    "Model": "LR",
    "Version": "Text-Only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.8399 seconds.
Total features =  500000
Accuracy:  0.49348401215011267
Precision:  0.4933289731850883
Recall:  0.9990728476821192
F1:  0.331732659755843

Confusion Matrix:
 [[   23 15494]
 [   14 15086]]

Top 5 Negative Features:
                   Feature  Importance
363351             reuters   -5.261452
1                      000   -3.034648
476308  washington reuters   -2.251313
164494              follow   -1.895461
87179                  com   -1.797532

Top 5 Positive Features:
              Feature  Importance
467873            via    3.280848
204051      image via    1.776547
393376          share    1.318721
296067        october    1.259612
292582  november 2016    1.232121


In [42]:
# LR Title-only
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = test_model(
    WWFND_raw, 
    vectorizer_title_lr_title_only, 
    vectorizer_text_lr_title_only, 
    model_lr_title_only,
    title_only=True
)
results.append({
    "Dataset": "WWFND",
    "Model": "LR",
    "Version": "Title-only",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

No of empty title rows:  30617
No of empty text rows:  1
No of rows with empty title AND text:  1

Features created in 0.1863 seconds.
Total features =  23007
Accuracy:  0.49319005781102004
Precision:  0.49319005781102004
Recall:  1.0
F1:  0.33029288885972397

Confusion Matrix:
 [[    0 15517]
 [    0 15100]]

Top 5 Negative Features:
          Feature  Importance
2596    breitbart   -6.310209
22909  york times   -4.528080
6976      factbox   -3.645147
5281   delingpole   -3.226715
16638    rohingya   -2.981946

Top 5 Positive Features:
        Feature  Importance
2542   breaking    3.703229
21764     video    3.483429
22783       wow    2.852267
564      actual    2.726850
22914     youll    2.682271


In [43]:
comparison_df = pd.DataFrame(results)
comparison_df.sort_values("Accuracy", ascending=False)

,Dataset,Model,Version,Accuracy,Precision,Recall,F1
7,IFND,LR,Text-only,0.666573,0.666549,1.000000,0.400169
5,IFND,LR,Baseline,0.666555,0.666537,1.000000,0.400112
6,IFND,LR,No-Source Words,0.666520,0.666614,0.999550,0.400859
3,IFND,SVM,Title-only,0.666502,0.666502,1.000000,0.399941
8,IFND,LR,Title-only,0.666502,0.666502,1.000000,0.399941
4,IFND,SVM,Hyper-Tuned,0.665374,0.666584,0.996243,0.403444
0,IFND,SVM,Baseline,0.664351,0.666348,0.994233,0.403952
2,IFND,SVM,Text-only,0.663593,0.666507,0.991243,0.407086
1,IFND,SVM,No-Source Words,0.658479,0.666456,0.976111,0.418155
9,WWFND,SVM,Baseline,0.501062,0.497047,0.980795,0.362374


In [44]:
comparison_df.to_csv(
    "../results/cross_dataset_validation.csv",
    index=False
)

In [45]:
for name, dataset in [
    ("WELFake", df_raw),
    ("IFND", IFND_raw),
    ("WWFND", WWFND_raw)
]:
    print(name)
    print(dataset["text"].str.split().str.len().describe())
    print()

WELFake
count    72095.000000
mean       540.843346
std        625.442464
min          0.000000
25%        227.000000
50%        399.000000
75%        667.000000
max      24234.000000
Name: text, dtype: float64

IFND
count    56714.000000
mean        12.923053
std          4.542416
min          2.000000
25%         10.000000
50%         12.000000
75%         15.000000
max         54.000000
Name: text, dtype: float64

WWFND
count    30616.000000
mean        18.416416
std          8.293947
min          1.000000
25%         12.000000
50%         17.000000
75%         24.000000
max         97.000000
Name: text, dtype: float64



## Final Analysis and Conclusions

The cross-dataset experiments revealed a significant gap between in-domain performance and real-world generalization.

While Linear SVM achieved nearly **98% accuracy** on the WELFake dataset, performance dropped to approximately **67% on IFND** and **50% on WWFND**. Similar behavior was observed across Logistic Regression and Naive Bayes, suggesting that the limitation was not primarily caused by classifier choice.

Several observations emerged from these experiments:

* Hyperparameter tuning produced only minor improvements and did not meaningfully improve generalization.
* Removing source identifiers reduced in-domain performance but had little effect on cross-dataset results.
* Text-only models generally performed better than title-only models, indicating that article content contains more useful information than headlines alone.
* Linear SVM remained the strongest overall model, although its advantage over Logistic Regression became much smaller on external datasets.

A likely explanation for the performance drop is the substantial difference between the datasets. WELFake contains long-form news articles with an average length of more than 500 words, while IFND and WWFND consist primarily of short news statements averaging fewer than 20 words. This creates a large vocabulary and distribution shift that TF-IDF based models struggle to handle.

Overall, the project demonstrates that classical machine learning methods can achieve very strong performance on a single dataset when combined with effective feature engineering. However, the cross-dataset evaluation highlights an important limitation: high accuracy on a benchmark dataset does not necessarily imply strong real-world robustness.

Future work could investigate domain adaptation techniques, semantic embeddings, and transformer-based approaches to improve generalization across different news sources and writing styles.
